# 逐交易日主力合约识别

**状态：Notebook 规划，尚未实现或运行识别算法。**

本方法面向熟悉 Python 和期货合约、首次进入本研究目录的读者。目标是根据预测时点已经可得的日线信息，为每个交易日选择主力合约，并保存可解释的选择依据。

输入是正式 silver 中的合约日线与日历；输出是一张逐交易日主力映射表。下游依据映射读取分钟行情，再独立处理收益与复权。

参考来源：

- [历史主力拼接 Notebook](../../../05_Old_Projects/old_02_Feature_Engineering/b01_main_continuous_merging.ipynb)：只读方法参考。
- [RB 主力分钟读取 Notebook](../../../00_draft_collection_01/testing_10.ipynb)：参考其信号日、生效日、候选排序与换约方向。
- [研究目录规则](../../AGENTS.md)：Notebook 定义、代理和实验成果归属。

下列规则均为拟实施方案。规划中的参数不表示已经完成数据覆盖检查或正式实验。

## 1. 输入与示例参数

| 输入 | 在主力识别中的作用 |
|---|---|
| 品种交易日历 | 确定相邻交易日，形成输出日期网格 |
| 合约日历 | 确定合约在生效日的交易资格及退市顺序 |
| 合约日线 | 提供信号日的成交量、持仓量及行情存在标记 |
| Bar 日历中的日线完成状态 | 区分尚未完成采集与已经完成但没有行情的情况 |

正式读取使用 `config.data_contracts` 中对应的权威 Arrow Schema，并按研究与湖仓规则展示、读取和转换。正式 silver 的已提交业务证明按契约使用。

首个示例拟使用：

| 参数 | 拟定值 |
|---|---|
| 交易所 | `XSGE` |
| 品种 | `RB` |
| 输出起始交易日 | `2015-01-01` |
| 输出结束交易日 | `2026-09-01` |
| 换约成交量倍数 | `1.10` |

**输出范围与识别历史不同。** 主力识别依赖此前持有的合约，必须从固定的历史初始化起点向后递推，再截取输出范围。实际读取历史、初始化日期和数据版本需要随结果保存。

## 2. 拟定的选择规则

### 时间顺序

信号交易日 $s$ 收盘后，利用 $s$ 的信息决定下一交易日 $t$ 的主力。输出分别记录 `signal_trading_date` 和 `trading_date`。

夜盘依照正式日历所属的交易日处理，不按时间戳的自然日期重新划分交易日。

### 初始化

候选合约须在信号日具有有效正成交量，且在生效日仍具交易资格。按以下顺序选出第一名：

1. 成交量降序。
2. 持仓量降序；缺失持仓量排在有效值之后。
3. 退市日期升序。
4. 合约代码升序。

没有有效候选时，保留未识别记录。初始化起点由固定的输入历史决定，不套用“至少 12 个在线合约”的条件。

### 正常换约

挑战者必须比现主力更晚退市。按同一候选排序选择挑战者，满足下式时换约：

$$V_{\mathrm{challenger},s}>1.10\,V_{\mathrm{incumbent},s}.$$

严格大于才触发；恰好等于门槛时保留现主力。倍数作为显式参数，首个示例固定为 1.10。

### 边界情况

| 情况 | 拟定处理 |
|---|---|
| 现主力有有效的零成交量 | 按数值零处理，可与有效挑战者直接比较 |
| 现主力行情或判断所需证据缺失 | 记录证据不足，不把缺失转换为零，也不据此自动换约 |
| 现主力在生效日已经不再交易 | 有有效后月候选时强制换约 |
| 现主力不再交易，且没有有效后月候选 | 保留未识别日期和原因 |
| 初始化或后续出现未识别日期 | 不静默跳过日期；保留最后已识别状态，后续仍遵守换约方向 |
| 日历缺少判断所必需的退市或交易资格信息 | 明确报告输入不足，避免猜测合约顺序 |

对“尚未采集完成”和“已经完成但没有行情”的具体状态分支，将在实现时依据权威字段语义明确展开。

## 3. 两份参考实现与正式方案的关系

| 事项 | 参考实现中的情况 | 本 Notebook 的规划 |
|---|---|---|
| 信号与生效日期 | 历史 Notebook 换约后将新合约写回当前日期的 `current_main`；testing_10 分开记录两个日期 | 显式建立信号日到下一交易日的映射 |
| 换约门槛 | 两份代码均使用严格超过 1.10 倍 | 保留该基准 |
| 换约方向 | 历史 Notebook 未限制后月；testing_10 限制更晚退市 | 保留只向更晚退市合约切换的设计 |
| 初始化 | 历史 Notebook 使用至少 12 个在线合约，并对个别品种截断日期 | 固定历史初始化条件并记录实际起点 |
| 零成交与缺失 | testing_10 先过滤非正成交量，导致现主力零成交与缺失进入同一后续分支 | 区分有效零值和缺失证据 |
| 无法识别的日期 | testing_10 的部分分支直接跳过该日期 | 每个目标交易日保留状态与原因 |

这些差异用于解释拟定方法的选择，不能据此宣称历史研究结果已经重新验证。

## 4. 计划中的代码与展示顺序

| 环节 | 待实现的单元格内容 | 可查看的结果 |
|---|---|---|
| 环境与契约 | 根目录定位、解释器、直接相关的权威 Schema 展示 | 输入表粒度、字段语义与运行环境 |
| 算法定义 | 一个连贯的逐交易日递推函数，标记 `export` | 输入、状态更新、输出的完整逻辑 |
| 示例参数 | 交易所、品种、输出范围、换约倍数 | 一份明确的运行参数 |
| 数据读取 | 显式读取日线、日历和相应完成状态 | 读取范围与输入覆盖 |
| 执行识别 | 显式调用算法，返回 `main_contract_daily_df` | 逐交易日的主力及决策依据 |
| 日期解释 | 展示初始化日、正常换约日与边界案例的候选表 | 人工核对为什么保留或换约 |
| 验证与汇总 | 时间顺序、门槛、换约方向、未识别日期和历史稳定性 | 换约明细、合约持有区间、诊断摘要 |
| 保存与复读 | 定义研究局部 Arrow Schema，写入 metadata 并复读 | 可供下游理解与追溯的结果 |

每个环节逐步加入代码。普通导入和算法定义可以标记 `export`；数据读取、执行、图表及保存均由明确的未标记单元格启动。

## 5. 每日主力映射的输出规划

主结果变量为 `main_contract_daily_df`，一行对应一个品种的一个目标交易日，包括未能识别主力的日期。

| 信息组 | 计划记录的字段或内容 |
|---|---|
| 业务身份 | `exchange_code`、`underlying_code` |
| 日期 | `trading_date`、`signal_trading_date` |
| 合约 | 最后已识别的现主力、有效挑战者、当日选定的主力合约 |
| 成交量证据 | 现主力、挑战者及最终选定合约在信号日的成交量 |
| 决策 | `selection_status`、`selection_reason`、`is_roll` |
| 参数 | `roll_trigger_ratio` |

未识别日期的最终主力为空；换约标记的可空规则及状态取值在实现局部 Schema 时一并确定。现主力字段须明确表示用于此次决策的历史状态，不能与上一行可能为空的选择结果混用。

结果 metadata 保存实际读取与输出范围、初始化起点、方法参数、输入摘要和代码身份。表格类型及可空约束由本方法的局部 Arrow Schema 定义。

## 6. 验证与成果归属

计划验证：

- 每个目标交易日有且仅有一条状态记录，信号交易日严格早于生效交易日。
- 被选合约在生效日具有交易资格，换约符合已经声明的退市顺序。
- 恰好达到 1.10 倍时不换约，严格超过时按规则换约。
- 成交量零值、缺失证据、到期无候选和并列排序分别有明确案例。
- 只改变输出起点时，重叠区间的选择保持一致。
- 追加或扰动截止日以后的输入时，截止日前的选择保持一致。

算法以本 Notebook 的 `export` 单元格为唯一来源。加入可导出定义后，再建立同名 `.py` 代理；代理只调用研究目录的 Notebook 加载器。

方法示例需要保存时使用本方法目录的 `demo.parquet`。正式实验直接调用同一定义，在所属时间分区的 `c01_Data` 生成输入；Hydra 配置和 MLflow Run 由实验入口统一管理。

**下一步实施内容：环境定位与输入契约展示，然后逐步加入识别算法。**